In [0]:
-- Qual categoria tem mais produtos vendidos? Até fim de 2025.

SELECT DescCategoriaProduto,
       count(*) AS qtdeProduto,
       sum(vlProduto) AS valorProduto

FROM workspace.tmw_points.transacoes AS t1

LEFT JOIN workspace.tmw_points.transacao_produto AS t2
ON t1.IdTransacao = t2.IdTransacao

LEFT JOIN workspace.tmw_points.produtos AS t3
ON t2.IdProduto = t3.IdProduto

WHERE t1.DtCriacao < '2026-01-01'
-- WHERE YEAR(t1.DtCriacao) <= 2025

GROUP BY DescCategoriaProduto
ORDER BY valorProduto DESC


In [0]:
-- Em 2024, quantas transações de Lovers tivemos?

SELECT count(*)

FROM workspace.tmw_points.transacoes AS t1

LEFT JOIN workspace.tmw_points.transacao_produto AS t2
ON t1.IdTransacao= t2.IdTransacao

LEFT JOIN workspace.tmw_points.produtos AS t3
ON t2.IdProduto = t3.IdProduto

WHERE t3.DescCategoriaProduto = 'lovers'
AND year(t1.DtCriacao) = 2024

In [0]:
-- Qual mês tivemos mais lista de presença assinada?

--  date_format(t1.DtCriacao, 'yyyy-MM') AS dtMes,
--  date(date_format(t1.DtCriacao, 'yyyy-MM-01')),
--  date_trunc('YEAR', t1.DtCriacao) AS dtAno,
--  date_trunc('DAY', t1.DtCriacao) AS dtDia,

SELECT 
      --  date(date_trunc('MONTH', t1.DtCriacao)) AS dtMes,
      --  date_format(t1.DtCriacao, 'yyyy-MM') AS dtMes,
       year(t1.DtCriacao) AS dtYear,
       month(t1.DtCriacao) AS dtMes,
       count(t1.IdTransacao) AS qtdeTransacoes

FROM workspace.tmw_points.transacoes AS t1

LEFT JOIN workspace.tmw_points.transacao_produto AS t2
ON t1.IdTransacao = t2.IdTransacao

LEFT JOIN workspace.tmw_points.produtos AS t3
ON t2.IdProduto = t3.IdProduto

WHERE t3.DescNomeProduto = 'Lista de presença'

GROUP BY dtYear, dtMes
ORDER BY qtdeTransacoes DESC

Databricks visualization. Run in Databricks to view.

In [0]:
-- Qual o total de pontos trocados no Stream Elements em Junho de 2025?

SELECT sum(t1.QtdePontos)

FROM workspace.tmw_points.transacoes AS t1

LEFT JOIN workspace.tmw_points.transacao_produto AS t2
ON t1.IdTransacao = t2.IdTransacao

LEFT JOIN workspace.tmw_points.produtos AS t3
ON t2.IdProduto = t3.IdProduto

WHERE year(t1.DtCriacao)=2025 AND month(t1.DtCriacao)=6
AND DescCategoriaProduto = 'streamelements'


In [0]:
-- 1. Quais clientes mais perderam pontos por Lover?

SELECT t1.IdCliente,
       sum(t1.QtdePontos) AS qtdePontosPerdidos

FROM workspace.tmw_points.transacoes AS t1

LEFT JOIN workspace.tmw_points.transacao_produto AS t2
ON t1.IdTransacao = t2.IdTransacao

LEFT JOIN workspace.tmw_points.produtos AS t3
ON t2.IdProduto = t3.IdProduto

WHERE DescCategoriaProduto = 'lovers'

GROUP BY t1.IdCliente
ORDER BY qtdePontosPerdidos ASC

In [0]:
-- 2. Quais clientes assinaram a lista de presença no dia 2025/08/25?


SELECT distinct IdCliente

FROM workspace.tmw_points.transacoes AS t1

LEFT JOIN workspace.tmw_points.transacao_produto AS t2
ON t1.IdTransacao = t2.IdTransacao

LEFT JOIN workspace.tmw_points.produtos AS t3
ON t2.IdProduto = t3.IdProduto

WHERE DescNomeProduto = 'Lista de presença'
AND date(t1.DtCriacao) = '2025-08-25'

In [0]:
-- Do início ao fim do nosso curso de SQL (2025/08/25 a 2025/08/29), quantos clientes assinaram a lista de presença?

SELECT count(distinct IdCliente) AS qtdeClientes

FROM workspace.tmw_points.transacoes AS t1

LEFT JOIN workspace.tmw_points.transacao_produto AS t2
ON t1.IdTransacao = t2.IdTransacao

LEFT JOIN workspace.tmw_points.produtos AS t3
ON t2.IdProduto = t3.IdProduto

WHERE DescNomeProduto = 'Lista de presença'
AND date(t1.DtCriacao) >= '2025-08-25'
AND date(t1.DtCriacao) <= '2025-08-29'

In [0]:
-- 4. Clientes mais antigos, tem mais quantidade de transação? Considere clientes mais antigos de 2025-12-31 e transações a partir de 2026-01-01.


WITH tb_clientes AS (
  SELECT idCliente,
         dtCriacao,
         date_diff('2025-12-31', DtCriacao) AS idadeDias
  FROM workspace.tmw_points.clientes
  WHERE date(DtCriacao) <= '2025-12-31'
),

tb_transacoes AS (
  SELECT IdCliente,
         count(*) AS qtdeTransacao
  FROM workspace.tmw_points.transacoes
  WHERE date(DtCriacao) >= '2026-01-01'
  GROUP BY IdCliente
)

SELECT t1.*,
       t2.qtdeTransacao


FROM tb_clientes AS t1

INNER JOIN tb_transacoes AS t2
ON t1.idcliente = t2.idcliente


Databricks visualization. Run in Databricks to view.

In [0]:
-- 5. Quantidade de transações Acumuladas ao longo do tempo do sistema?

WITH daily_report AS (

  SELECT date(DtCriacao) AS dtDia,
        count(*) AS qtdeTransacao

  FROM workspace.tmw_points.transacoes
  GROUP BY dtDia

)

SELECT *,
       SUM(qtdetransacao) OVER (ORDER BY dtdia) AS acumQtdetransacao

FROM daily_report

Databricks visualization. Run in Databricks to view.

In [0]:
-- 6. Quantidade de usuários cadastrados (absoluto e acumulado) ao longo do tempo?

WITH daily_report AS (
    SELECT date(DtCriacao) AS dtDia,
          count(*) AS qtdeCadastros
    FROM workspace.tmw_points.clientes AS t1
    GROUP BY dtdia
)

SELECT *,
       sum(qtdecadastros) OVER (ORDER BY dtdia) AS qtdeCadastrosAcum
FROM daily_report

In [0]:
-- 7. Qual o dia da semana mais ativo de cada usuário?

WITH tb_cliente_dia_semana AS (

  SELECT IdCliente,
        weekday(DtCriacao) AS diaSemana,
        count(*) AS qtdeTransacoes
  FROM workspace.tmw_points.transacoes
  GROUP BY ALL

)

SELECT *
FROM tb_cliente_dia_semana
QUALIFY row_number() OVER (PARTITION BY idCliente ORDER BY qtdeTransacoes DESC) = 1

In [0]:
-- 8. Saldo de pontos acumulado de cada usuário
WITH tb_cliente_dia AS (
    SELECT IdCliente,
          date(dtCriacao) AS dtDia,
          sum(QtdePontos) AS qtdePontos
    FROM workspace.tmw_points.transacoes
    GROUP BY ALL
)

SELECT *,
       sum(qtdePontos) OVER (PARTITION BY idCliente ORDER BY dtdia) AS totalAcum,
       row_number() OVER (PARTITION BY idCliente ORDER BY dtdia) AS ordemDia
 
FROM tb_cliente_dia

In [0]:
-- 9. Dos clientes que começaram SQL no primeiro dia (2025/08/25), quantos chegaram ao 5o dia (2025/08/29)?

WITH tb_cliente_inicio AS (

    SELECT distinct IdCliente
    FROM workspace.tmw_points.transacoes
    WHERE date(DtCriacao) = '2025-08-25'

),

tb_cliente_fim AS (

    SELECT distinct IdCliente
    FROM workspace.tmw_points.transacoes
    WHERE date(DtCriacao) = '2025-08-29'

)

SELECT count(t1.idcliente) AS qtdeClientesInicio,
       count(t2.idcliente)  AS qtdeCLienteFim,
       count(t2.idcliente) / count(t1.idcliente) AS retencaoFinal

FROM tb_cliente_inicio AS t1

LEFT JOIN tb_cliente_fim AS t2
ON t1.idCliente = t2.idCliente

In [0]:
-- 10. Como foi a curva de Churn do Curso de SQL? Considere Churn diário.
WITH tb_cliente_inicio AS (

    SELECT distinct IdCliente
    FROM workspace.tmw_points.transacoes
    WHERE date(DtCriacao) = '2025-08-25'

),

tb_dia_atividade AS (

    SELECT DISTINCT IdCliente, date(DtCriacao) AS dtDia
    FROM workspace.tmw_points.transacoes
    WHERE date(DtCriacao) >= '2025-08-25'
    AND date(DtCriacao) <= '2025-08-29'

)

SELECT dtdia,
       count(*) AS tqdeDia -- retidos
FROM tb_cliente_inicio AS t1
LEFT JOIN tb_dia_atividade AS t2
ON t1.idcliente = t2.idcliente

GROUP BY 1
ORDER BY 1

Databricks visualization. Run in Databricks to view.

In [0]:
-- 11. Quem iniciou o curso no primeiro dia, em média assistiu quantas aulas?

WITH tb_cliente_inicio AS (

    SELECT distinct IdCliente
    FROM workspace.tmw_points.transacoes
    WHERE date(DtCriacao) = '2025-08-25'

),

tb_dia_atividade AS (

    SELECT DISTINCT IdCliente, date(DtCriacao) AS dtDia
    FROM workspace.tmw_points.transacoes
    WHERE date(DtCriacao) >= '2025-08-25'
    AND date(DtCriacao) <= '2025-08-29'

),

qtde_dias_cliente AS (

    SELECT t1.idcliente,
          count(distinct dtDia) AS qtdeDias
    FROM tb_cliente_inicio AS t1

    LEFT JOIN tb_dia_atividade AS t2
    ON t1.idcliente = t2.idcliente

    GROUP BY t1.idcliente
    ORDER BY t1.idcliente

)


SELECT avg(qtdeDias) AS mediaDiasAssistidos,
       median(qtdedias) AS medianaDiasAssistidos

FROM qtde_dias_cliente

In [0]:
-- 12. Dentre os clientes de janeiro/2025, quantos assistiram o curso de SQL?

WITH tb_cliente_jan2025 AS (
    SELECT distinct idCliente
    FROM workspace.tmw_points.transacoes
    WHERE year(DtCriacao) = 2025
    AND month(DtCriacao) = 1
),

tb_clientes_curso (
    
    SELECT DISTINCT IdCliente
    FROM workspace.tmw_points.transacoes
    WHERE date(DtCriacao) >= '2025-08-25'
    AND date(DtCriacao) <= '2025-08-29'

)


SELECT count(t1.idcliente) AS qtdeClienteJan2025,
       count(t2.idcliente) AS qtdeClienteJan2025Curso,
       count(t2.idcliente) / count(t1.idcliente) AS pctClienteJan2025Curso

FROM tb_cliente_jan2025 AS t1

LEFT JOIN tb_clientes_curso AS t2
ON t1.idcliente = t2.idcliente

In [0]:
-- 13. Qual o dia de curso com maior engajamento de cada aluno que iniciou o curso no dia 01 (2025/08/25)?

WITH tb_cliente_inicio AS (

    SELECT distinct IdCliente
    FROM workspace.tmw_points.transacoes
    WHERE date(DtCriacao) = '2025-08-25'

),

tb_transacao AS (

    SELECT idCliente,
           date(dtCriacao) as diaCurso,
           count(*) AS qtdeTransacao
    
    FROM workspace.tmw_points.transacoes
    
    WHERE date(DtCriacao) >= '2025-08-25'
    AND date(DtCriacao) <= '2025-08-29'

    GROUP BY ALL
)

SELECT *
FROM tb_cliente_inicio AS t1

LEFT JOIN tb_transacao AS t2
ON t1.idCliente = t2.idcliente

QUALIFY row_number() OVER (PARTITION BY t1.idCliente ORDER BY qtdeTransacao DESC) = 1